# High-Water — a quantitative teardown 🔬
### The nearness hedge with a Lo t-stat · the 0.82 correlation to 12-2 momentum · the survivor-sign caveat · costs

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![A distinct anomaly?: Busted](https://img.shields.io/badge/A_distinct_anomaly%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We test whether the 52-week-high effect pays and whether it's distinct from momentum.

> ⚠️ **Not investment advice.** 398 *current* S&P 500 names with ≥20y history (Yahoo), 1996–2026 — a **survivor panel** (`fetch_panel` requires an explicit `allow_survivorship_bias=True`), and here the bias is load-bearing: the short leg holds fallen names guaranteed to have survived, so the hedge's negative *level* is partly an artifact. The momentum *correlation* is the bias-insensitive statistic. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (high_water/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from high_water import data, strategy as st
# Opt-in, stated: current S&P 500 members + a >=20y-history filter = a survivor panel. Here the
# bias hits the SIGN: the short leg (fallen names) is guaranteed survivors, so the negative hedge
# is partly manufactured. The momentum correlation is the bias-robust number.
ret = data.fetch_panel(allow_survivorship_bias=True)   # cache-first; built by examples/verify.py --fetch
hi = st.cross_section_hedge(ret, st.nearness(ret))
mo = st.cross_section_hedge(ret, st.momentum(ret))     # standard 12-2 momentum (skips the last month)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | no positive premium; the measured −8.4%/yr (t −2.2) is partly the survivor panel's manufacture |
| Tradability | **Mirage** | nothing gross to keep; one-way costs (3.2× NAV/mo) dig |
| Distinct anomaly? | **Busted** | 0.82 correlated with standard 12-2 momentum — bias-robust |

> 💡 *In plain words:* momentum relabelled, and no premium on top — the half of the verdict that doesn't depend on our flawed panel is the correlation, and it's damning enough.

## 1 · The claim, steelmanned

- **H₁:** the nearness long-short earns a significant positive premium.
- **H₂:** it is distinct from 12-2 momentum (low correlation).
- **H₃:** it's stable over time.

**Known bias, named up front:** current membership × a ≥20y-history filter = survivors. For H₂ (a correlation between two hedges run on the same panel) the bias is near-neutral; for H₁'s *level* it is not — it pushes the short-the-fallen book down, i.e. the hedge's sign toward negative.

## 2 · So what? — what rides on each

If H₁/H₂ hold, it's a new tradable factor. If H₂ fails, it's momentum with a story; if H₁ fails too, there's no premium at all.

## 3 · How we'd know — the protocol

Nearness long-short + Lo t-stat → correlation with the standard 12-2 momentum hedge (trailing year, skipping the most recent month) → decade split → one-way cost sweep.

## 4 · The teardown

### 4.1 The nearness hedge vs the 12-2 momentum control

In [2]:
display(pd.DataFrame({'52-week-high':st.stats(hi),'12-2 momentum':st.stats(mo)}).T[['mean_ann','sharpe','tstat','hit_rate','n']].round(3))
print(f"corr(nearness hedge, 12-2 momentum hedge) = {st.signal_overlap(ret):+.2f}")
mo_naive = st.cross_section_hedge(ret, st.momentum(ret, skip=0))   # includes the last month
import pandas as _pd
print(f"corr vs naive trailing-12 (incl. last month): "
      f"{_pd.concat([hi.rename('h'),mo_naive.rename('m')],axis=1).dropna().corr().iloc[0,1]:+.2f}")

,mean_ann,sharpe,tstat,hit_rate,n
52-week-high,-0.084,-0.401,-2.206,0.505,366.0
12-2 momentum,0.015,0.072,0.399,0.553,365.0


corr(nearness hedge, 12-2 momentum hedge) = +0.82


corr vs naive trailing-12 (incl. last month): +0.87


> 💡 *In plain words:* nearness shows no premium (−8.4%/yr, t −2.2 — read the sign with the survivor caveat) and is **0.82 correlated** with standard 12-2 momentum (0.87 against the naive trailing-12 — the redundancy doesn't hinge on the momentum convention). **H₁ rejected (no premium), H₂ rejected (same factor).**

### 4.2 Decay

In [3]:
for lab,sl in [('1996-2012',hi.loc[:'2012']),('2013-on',hi.loc['2013':])]:
    print(f'{lab}: Sharpe {st.stats(sl)["sharpe"]:+.2f}, mean {st.stats(sl)["mean_ann"]:+.2%}/yr')

1996-2012: Sharpe -0.52, mean -13.01%/yr
2013-on: Sharpe -0.18, mean -2.64%/yr


> 💡 *In plain words:* strongly negative early — the 2000/2008 rebounds, where the survivor panel's guaranteed comebacks hit the short leg hardest — drifting to zero since. **H₃** — no stable positive regime anywhere.

### 4.3 Cost sweep — one-way, and a non-positive book only gets worse

In [4]:
rows={'gross':st.stats(hi)['sharpe']}
for c in (5,10,20): rows[f'{c}bp']=st.stats(st.net_of_cost(hi,c))['sharpe']
display(pd.Series(rows, name='net Sharpe (one-way bp)').round(3))

gross   -0.401
5bp     -0.492
10bp    -0.583
20bp    -0.766
Name: net Sharpe (one-way bp), dtype: float64

> 💡 *In plain words:* replacing ~80% of a two-sided book monthly is ~3.2× NAV of one-way trades; with nothing positive to start from, costs just compound the loss.

## 5 · The verdict

H₁ rejected (no premium — the negative level partly panel-made, the absence of a positive one not), H₂ rejected (0.82 with momentum, bias-robust), H₃ rejected → Signal `NONE`, Tradability `MIRAGE`, distinct-anomaly claim `BUSTED`.

## 6 · Could you trade it?

No — and if you wanted the momentum exposure it proxies, you'd use momentum directly (cleaner, and itself fragile on large caps — Study 24 Stampede). The 52-week-high label adds a narrative, not an edge.

## 7 · Going further

Forks: (a) a point-in-time universe — the single change that would let the hedge's *level* be read at face value (and the panel guard dropped); (b) double-sort nearness × momentum to confirm there's no residual; (c) the momentum-crash overlay (Daniel-Moskowitz 2016) on the near-high book. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).